# FeedSure 360 — Computer Vision & Chemical Strip Training Pipeline
**SIH 2026 Problem Statement 26111** | Ministry of Fisheries, Animal Husbandry & Dairying

> **Module:** Adaptive Camera Screening & Rapid Chemical Test Colorimeter
> **Purpose:** Trains visual feed type classifier, surface mould anomaly detector, and urea test strip colorimeter.


## 1. Setup & Environment
Install required dependencies for Google Colab / local environment.

In [ ]:
# !pip install numpy pandas scikit-learn pillow scipy joblib matplotlib

## 2. Load / Generate Datasets
- **Dataset D**: Feed Surface & Texture Dataset (Maize Silage, Green Fodder, Dry Fodder, Concentrate, and Mould Anomalies)
- **Dataset E**: Urea Test Strip Colorimetric Calibration Dataset (Bromothymol Blue + Urease reaction)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Add backend directory to sys.path
sys.path.append("..")
from ml.dataset_generator import generate_feed_surface_dataset, generate_urea_strip_dataset, FEATURE_NAMES

print("Generating reference datasets...")
df_feed, df_anomaly = generate_feed_surface_dataset(samples_per_class=200, seed=42)
df_urea = generate_urea_strip_dataset(num_samples=600, seed=42)
print(f"Feed Surface Dataset: {len(df_feed)} clean samples, {len(df_anomaly)} total anomaly samples.")
print(f"Urea Test Strip Dataset: {len(df_urea)} calibrated colorimetry samples.")
df_feed.head()

## 3. Train Model 1 — Visual Feed Classifier
Classifies feed types (Maize Silage, Green Fodder, Dry Fodder, Concentrate) from 22 color and texture features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

X = df_feed[FEATURE_NAMES].values
y = df_feed["feed_class_idx"].values
feed_names = ["Maize Silage", "Green Fodder", "Dry Fodder", "Concentrate"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf_feed = RandomForestClassifier(n_estimators=120, max_depth=10, random_state=42)
clf_feed.fit(X_train_scaled, y_train)

y_pred = clf_feed.predict(X_test_scaled)
print("Feed Classifier Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=feed_names))

## 4. Train Model 2 — Surface Mould Anomaly Detector
Identifies atypical surface discoloration, mycelium clusters, or anaerobic dark spoilage.

In [ ]:
X_anom = df_anomaly[FEATURE_NAMES].values
y_anom = df_anomaly["is_anomaly"].values

X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(X_anom, y_anom, test_size=0.25, random_state=42, stratify=y_anom)
a_scaler = StandardScaler()
X_a_train_scaled = a_scaler.fit_transform(X_a_train)
X_a_test_scaled = a_scaler.transform(X_a_test)

clf_anomaly = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight="balanced", random_state=42)
clf_anomaly.fit(X_a_train_scaled, y_a_train)

y_a_pred = clf_anomaly.predict(X_a_test_scaled)
print("Anomaly Detector Accuracy:", accuracy_score(y_a_test, y_a_pred))
print(classification_report(y_a_test, y_a_pred, target_names=["Normal Feed", "Mould/Spoilage Anomaly"]))

## 5. Train Model 3 — Rapid Urea Paper Strip Colorimeter
Predicts urea concentration and flags adulteration risk from strip RGB / CIELab / HSV features.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

color_features = ["hue_deg", "saturation", "value", "r", "g", "b", "L", "a", "b_cielab"]
X_u = df_urea[color_features].values
y_u_reg = df_urea["urea_concentration_pct"].values
y_u_clf = df_urea["risk_index"].values

X_u_train, X_u_test, y_u_train, y_u_test = train_test_split(X_u, y_u_reg, test_size=0.25, random_state=42)
u_scaler = StandardScaler()
X_u_train_scaled = u_scaler.fit_transform(X_u_train)
X_u_test_scaled = u_scaler.transform(X_u_test)

reg_urea = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
reg_urea.fit(X_u_train_scaled, y_u_train)
y_u_pred = reg_urea.predict(X_u_test_scaled)
print("Urea Regression R2 Score:", r2_score(y_u_test, y_u_pred))

## 6. Export Models for FeedSure 360 Backend
Persists the model binaries into `backend/models/vision/` for the FastAPI backend service.

In [ ]:
from ml.train_vision_models import run_full_training

# Run full automated training pipeline and save artifacts
run_full_training()